In [149]:
import sympy as sp
from sympy.physics.vector import dynamicsymbols
from sympy import Eq

# System state components
p_x, p_y, p_z = dynamicsymbols('p_x p_y p_z')           # position components    WORLD FRAME
v_x, v_y, v_z = dynamicsymbols('v_x v_y v_z')           # velocity components    WORLD FRAME
q_w, q_x, q_y, q_z = dynamicsymbols('q_w q_x q_y q_z')  # quaternion components  WORLD FRAME
w_x, w_y, w_z = dynamicsymbols('w_x w_y w_z')           # angular rates          BODY FRAME
p = sp.Matrix([p_x, p_y, p_z])
q = sp.Matrix([q_w, q_x, q_y, q_z])
v = sp.Matrix([v_x, v_y, v_z])
w = sp.Matrix([w_x, w_y, w_z])
x = sp.Matrix([p, q, v, w])
x

Matrix([
[p_x(t)],
[p_y(t)],
[p_z(t)],
[q_w(t)],
[q_x(t)],
[q_y(t)],
[q_z(t)],
[v_x(t)],
[v_y(t)],
[v_z(t)],
[w_x(t)],
[w_y(t)],
[w_z(t)]])

In [150]:
# System input components
omega0, omega1, omega2, omega3 = dynamicsymbols('Omega0 Omega1 Omega2 Omega3')  # speeds of each rotor BODY FRAME
omega = sp.Matrix([omega0, omega1, omega2, omega3])
omega

Matrix([
[Omega0(t)],
[Omega1(t)],
[Omega2(t)],
[Omega3(t)]])

In [151]:
# System parameters
I_xx, I_yy, I_zz, I_xy, I_xz, I_yz = sp.symbols('I_xx I_yy I_zz I_xy I_xz I_yz')  # moments of inertia
J = sp.Matrix([[I_xx, -I_xy, -I_xz],
                [-I_xy, I_yy, -I_yz],
                [-I_xz, -I_yz, I_zz]])
m, g = sp.symbols('m g')  # mass, gravity
d = sp.symbols('d')  # distance from center to each rotor
r0 = sp.Matrix([-d, -d, 0])  # position of rotor 0 in BODY FRAME
r1 = sp.Matrix([d, -d, 0])  # position of rotor 1 in BODY FRAME
r2 = sp.Matrix([d, d, 0])  # position of rotor 2 in BODY FRAME
r3 = sp.Matrix([-d, d, 0])  # position of rotor 3 in BODY FRAME
Cl, Cd = sp.symbols('C_l C_d')  # lift and drag coefficients of each rotor

#### In these section there are motion equations for each component of state vector. Note, they are differentially flat.

In [152]:

p_eq = Eq(p.diff(), v)
p_eq

Eq(Matrix([
[Derivative(p_x(t), t)],
[Derivative(p_y(t), t)],
[Derivative(p_z(t), t)]]), Matrix([
[v_x(t)],
[v_y(t)],
[v_z(t)]]))

In [153]:
def omega_matrix(w):
    w_x, w_y, w_z = w
    return sp.Matrix([[0, -w_x, -w_y, -w_z],
                      [w_x, 0, w_z, -w_y],
                      [w_y, -w_z, 0, w_x],
                      [w_z, w_y, -w_x, 0]])

q_eq = Eq(q.diff(), 0.5 * omega_matrix(w) * q)
q_eq

Eq(Matrix([
[Derivative(q_w(t), t)],
[Derivative(q_x(t), t)],
[Derivative(q_y(t), t)],
[Derivative(q_z(t), t)]]), Matrix([
[-0.5*q_x(t)*w_x(t) - 0.5*q_y(t)*w_y(t) - 0.5*q_z(t)*w_z(t)],
[ 0.5*q_w(t)*w_x(t) + 0.5*q_y(t)*w_z(t) - 0.5*q_z(t)*w_y(t)],
[ 0.5*q_w(t)*w_y(t) - 0.5*q_x(t)*w_z(t) + 0.5*q_z(t)*w_x(t)],
[ 0.5*q_w(t)*w_z(t) + 0.5*q_x(t)*w_y(t) - 0.5*q_y(t)*w_x(t)]]))

In [154]:
F_g = sp.Matrix([0, 0, -m * g])  # gravity force in WORLD FRAME
F_bf = Cl * sp.matrix_multiply_elementwise(omega, omega)  # thrust produced by each rotor
F_total_bf = sp.Matrix([0, 0, sum(f for f in F_bf)])  # total thrust in BODY FRAME
R = sp.MatrixSymbol('R(q)_wb', 3, 3)  # rotation matrix from BODY to WORLD FRAME
F_total_wf = R * F_total_bf + F_g  # total force in WORLD FRAME
Eq(v.diff(), F_total_wf / m)  # acceleration equation


Eq(Matrix([
[Derivative(v_x(t), t)],
[Derivative(v_y(t), t)],
[Derivative(v_z(t), t)]]), 1/m*(Matrix([
[   0],
[   0],
[-g*m]]) + R(q)_wb*Matrix([
[                                                                        0],
[                                                                        0],
[C_l*Omega0(t)**2 + C_l*Omega1(t)**2 + C_l*Omega2(t)**2 + C_l*Omega3(t)**2]])))

In [ ]:
tau_prop = Cd * sp.matrix_multiply_elementwise(omega, omega)  # drag torque produced by each rotor
tau_prop[1] = -tau_prop[1]  # rotor 1 spins opposite direction
tau_prop[3] = -tau_prop[3]  # rotor 3 spins opposite direction
tau_bw = (r0.cross(sp.Matrix([0, 0, F_bf[0]])) +
          r1.cross(sp.Matrix([0, 0, F_bf[1]])) +
          r2.cross(sp.Matrix([0, 0, F_bf[2]])) +
          r3.cross(sp.Matrix([0, 0, F_bf[3]])))
tau_bw[2] += tau_prop[0] + tau_prop[1] + tau_prop[2] + tau_prop[3]
Eq(w.diff(), J.inv() * (tau_bw - w.cross(J * w)))

Eq(Matrix([
[Derivative(w_x(t), t)],
[Derivative(w_y(t), t)],
[Derivative(w_z(t), t)]]), Matrix([
[(I_xy*I_yz + I_xz*I_yy)*(C_d*Omega0(t)**2 - C_d*Omega1(t)**2 + C_d*Omega2(t)**2 - C_d*Omega3(t)**2)/(I_xx*I_yy*I_zz - I_xx*I_yz**2 - I_xy**2*I_zz - 2*I_xy*I_xz*I_yz - I_xz**2*I_yy) + (I_xy*I_zz + I_xz*I_yz)*(C_l*d*Omega0(t)**2 - C_l*d*Omega1(t)**2 - C_l*d*Omega2(t)**2 + C_l*d*Omega3(t)**2)/(I_xx*I_yy*I_zz - I_xx*I_yz**2 - I_xy**2*I_zz - 2*I_xy*I_xz*I_yz - I_xz**2*I_yy) + (I_yy*I_zz - I_yz**2)*(-C_l*d*Omega0(t)**2 - C_l*d*Omega1(t)**2 + C_l*d*Omega2(t)**2 + C_l*d*Omega3(t)**2)/(I_xx*I_yy*I_zz - I_xx*I_yz**2 - I_xy**2*I_zz - 2*I_xy*I_xz*I_yz - I_xz**2*I_yy)],
[(I_xx*I_yz + I_xy*I_xz)*(C_d*Omega0(t)**2 - C_d*Omega1(t)**2 + C_d*Omega2(t)**2 - C_d*Omega3(t)**2)/(I_xx*I_yy*I_zz - I_xx*I_yz**2 - I_xy**2*I_zz - 2*I_xy*I_xz*I_yz - I_xz**2*I_yy) + (I_xx*I_zz - I_xz**2)*(C_l*d*Omega0(t)**2 - C_l*d*Omega1(t)**2 - C_l*d*Omega2(t)**2 + C_l*d*Omega3(t)**2)/(I_xx*I_yy*I_zz - I_xx*I_yz**2 - I_xy**2*I_zz -

: 